In [ ]:
# ===== Notebook cell 1 =====
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("AReM")
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
for activity in activities:
    files = list((data_path / activity).glob("*.csv"))
    print(f"Activity: {activity}, Number of files: {len(files)}")

# ===== Notebook cell 2 =====
#1b 
train_files = []
test_files = []
for activity in activities:
    folder = data_path / activity
    files = sorted(folder.glob("*.csv"),
                   key=lambda f: int(f.stem.replace("dataset", "")))

    if activity in ["bending1", "bending2"]:
        test_files.extend(files[:2])
        train_files.extend(files[2:])
    else:
        test_files.extend(files[:3])
        train_files.extend(files[3:])
print("Training files:", len(train_files))
print("Testing files:", len(test_files))

# figuring out shape
sample_file = train_files[0]
df = pd.read_csv(sample_file, comment="#",header=None)
print("File", sample_file)
print("Shape", df.shape)
display(df.head())

# ===== Notebook cell 3 =====

def load_arem_file(file):
    # Read the file and remove trailing commas
    with open(file, "r") as f:
        lines = [
            line.rstrip("\r\n").rstrip(",")
            for line in f
            if line.strip() and not line.lstrip().startswith("#")
        ]

    # Handle comma-separated and whitespace-separated files
    from io import StringIO

    text = "\n".join(lines)

    if "," in lines[0]:
        df = pd.read_csv(StringIO(text), header=None)
    else:
        df = pd.read_csv(StringIO(text), header=None, sep=r"\s+")

    return df

# ===== Notebook cell 4 =====
#1c. ii
sensor_data = df.iloc[:, 1:]
print("Sensor data shape:", sensor_data.shape)
features = sensor_data.agg(['mean', 'std', 'min', 'max', 'median'])
print(features)

#q1, q3
q1 = sensor_data.quantile(0.25)
q3 = sensor_data.quantile(0.75)
print("Q1:", q1)    
print("Q3:", q3)

features.loc["q1"] = q1
features.loc["q3"] = q3
display(features)

feature_rows= []
for file in train_files + test_files:
    df = load_arem_file(file)
    sensor_data = df.iloc[:, 1:]
    row = {}
    for i, col in enumerate(sensor_data.columns, start=1):
        values = sensor_data[col]
        row[f"sensor_{i}_mean"] = values.mean()
        row[f"sensor_{i}_std"] = values.std()
        row[f"sensor_{i}_min"] = values.min()
        row[f"sensor_{i}_max"] = values.max()
        row[f"sensor_{i}_median"] = values.median()
        row[f"sensor_{i}_q1"] = values.quantile(0.25)
        row[f"sensor_{i}_q3"] = values.quantile(0.75)
    feature_rows.append(row)
feature_df = pd.DataFrame(feature_rows)
print("Feature DataFrame shape:", feature_df.shape)
display(feature_df.head())
print(feature_df.shape) # confirming shape is good to go

# ===== Notebook cell 5 =====
# 1ciii std and 90% bootstrap CI 
feature_std = feature_df.std()
display(feature_std)

import numpy as np
np.random.seed(42) 
n_bootstrap = 1000
bootstrap_stds = []

for i in range(n_bootstrap):
    sample = feature_df.sample(n=len(feature_df), replace=True)
    bootstrap_stds.append(sample.std().values)

bootstrap_stds = np.array(bootstrap_stds)
lower = np.percentile(bootstrap_stds, 5, axis=0)
upper = np.percentile(bootstrap_stds, 95, axis=0)  

std_results = pd.DataFrame({
    "Feature": feature_df.columns,
    "Standard Deviation": feature_df.std().values,
    "90% CI Lower": lower,
    "90% CI Upper": upper})

display(std_results.round(3))

# ===== Notebook cell 8 =====
# question 4

train_features = feature_df.iloc[:len(train_files)].copy()
train_labels = [ 1 if file.parent.name in ["bending1", "bending2"] else 0 for file in train_files]
train_features['class'] = train_labels
print("Training data shape:", train_features.shape)
print("Class distribution:")
print(train_features['class'].value_counts())


#selected features (mean, std, max)
selected_features = []
for i in [1, 2, 6]:
    selected_features.extend([
        f"sensor_{i}_mean",
        f"sensor_{i}_std",
        f"sensor_{i}_max" ])
plot_data = train_features[selected_features + ["class"]]
display(plot_data.head())

import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(plot_data, vars=selected_features, hue="class", palette={0: "blue", 1: "orange"}, diag_kind="hist")
plt.show()

# ===== Notebook cell 9 =====
#4aii

split_rows = []
for file in train_files:
    df = load_arem_file(file)
    sensor_data = df.iloc[:, 1:]
    row = {}

    for i, col in enumerate(sensor_data.columns, start=1):
        values = sensor_data[col]
        segments = np.array_split(values.to_numpy(), 2)

        for j, segment in enumerate(segments, start=1):
            row[f"sensor_{i}_part{j}_mean"] = np.mean(segment)
            row[f"sensor_{i}_part{j}_std"] = np.std(segment, ddof=1)
            row[f"sensor_{i}_part{j}_max"] = np.max(segment)
    row["class"] = (
        1 if file.parent.name in ["bending1", "bending2"] else 0)
    split_rows.append(row)
split_features = pd.DataFrame(split_rows)
print("Shape:", split_features.shape)
display(split_features.head())

# scatterplot portion
selected_split_features = []
for i in [1, 2, 6]:
    for j in [1, 2]:
        selected_split_features.extend([
            f"sensor_{i}_part{j}_mean",
            f"sensor_{i}_part{j}_std",
            f"sensor_{i}_part{j}_max"])
sns.pairplot(split_features, vars= selected_split_features, hue="class", palette={0: "blue", 1: "orange"}, diag_kind="hist")

# ===== Notebook cell 11 =====
#4a iii
def extract_segment_features(files, l):
    rows = []
    labels = []
    for file in files:
        df = load_arem_file(file)
        sensor_data = df.iloc[:, 1:]
        row = {}
        for i, col in enumerate(sensor_data.columns, start=1):
            values = sensor_data[col]
            segments = np.array_split(values.to_numpy(), l)
            for j, segment in enumerate(segments, start=1):
                row[f"sensor_{i}_part{j}_min"] = np.min(segment)
                row[f"sensor_{i}_part{j}_max"] = np.max(segment)
                row[f"sensor_{i}_part{j}_mean"] = np.mean(segment)
                row[f"sensor_{i}_part{j}_median"] = np.median(segment)
                row[f"sensor_{i}_part{j}_std"] = np.std(segment, ddof=1)
                row[f"sensor_{i}_part{j}_q1"] = np.percentile(segment, 25)
                row[f"sensor_{i}_part{j}_q3"] = np.percentile(segment, 75)
        rows.append(row)
        labels.append(1 if file.parent.name in ["bending1", "bending2"] else 0)
    X = pd.DataFrame(rows)
    y = pd.Series(labels)
    return X, y


# training and testing function 
X_train1, y_train1 = extract_segment_features(train_files, 1)
X_train2, y_train2 = extract_segment_features(train_files, 2)
print("l=1", X_train1.shape)
print("l=2", X_train2.shape)
print("Labels:", np.bincount(y_train1))

# setup
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)
print(cv)


results = []
p_values = [1,2,3,5,10,15,20]

for l in range(1,21):
    X,y = extract_segment_features(train_files, l)
    for p in p_values:
        if p > X.shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, C=1e6),
                n_features_to_select=p,
                step=0.2 )),
            ("logistic", LogisticRegression(
                max_iter=1000,
                C=1e6 ))  ])

        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="accuracy" )

        results.append({
            "l": l,
            "p": p,
            "CV Accuracy": scores.mean(),
            "CV Std": scores.std() })

cv_results = pd.DataFrame(results)
display(
    cv_results.sort_values(
        "CV Accuracy",
        ascending=False ).head(10))

best_result = cv_results.sort_values(
    ["CV Accuracy", "p", "l"],
    ascending=[False, True, True]
).iloc[0]

best_l = int(best_result["l"])
best_p = int(best_result["p"])
print(f"Best l: {best_l}, Best p: {best_p}")
print("Best CV Accuracy:", round(best_result["CV Accuracy"], 4))

# ===== Notebook cell 13 =====
#4a iv
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

X_train, y_train = extract_segment_features(train_files, best_l)
best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, C=1e6),
        n_features_to_select=best_p,
        step=0.2 )),
    ("logistic", LogisticRegression(
        max_iter=1000,
        C=1e6 ))  ])

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_train)
y_prob = best_model.predict_proba(X_train)[:,1]

print("Confusion Matrix:")
print(confusion_matrix(y_train, y_pred))
print("Training AUC:", roc_auc_score(y_train, y_prob))
fpr, tpr, thresholds = roc_curve(y_train, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_train, y_prob):.4f}")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Training ROC Curve")
plt.legend()
plt.show()


#4a iv coefficients
selected_features = X_train.columns[best_model.named_steps["rfe"].support_]
coefficients = best_model.named_steps["logistic"].coef_[0]
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": coefficients})
display(coef_df)

# p-values
import statsmodels.api as sm
X_scaled = best_model.named_steps["scaler"].transform(X_train)
X_selected = X_scaled[:, best_model.named_steps["rfe"].support_]
X_sm = sm.add_constant(X_selected)
logit_model = sm.Logit(y_train, X_sm)
logit_result = logit_model.fit(disp=False)
print(logit_result.summary())

# ===== Notebook cell 15 =====
# 4a v
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
X_test, y_test = extract_segment_features(test_files, best_l)
y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:,1]
test_accuracy = accuracy_score(y_test, y_test_pred)
test_error = 1 - test_accuracy
print("test accuracy:", round(test_accuracy, 4))
print("test error:", round(test_error, 4))  
print("test confusion matrix: ")
print(confusion_matrix(y_test, y_test_pred))
print("test AUC:", round(roc_auc_score(y_test, y_test_prob), 4))

# ===== Notebook cell 17 =====
#4b 
from sklearn.model_selection import GridSearchCV
l1_results = []
C_values = [0.001,0.01,0.1,1,10,100]
for l in range(1,21):
    X,y = extract_segment_features(train_files, l)
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=1000 ))  ])
    grid = GridSearchCV(
        model,
        {"logistic__C": C_values},
        cv=cv,
        scoring="accuracy" )
    grid.fit(X,y)
    l1_results.append({
        "l": l,
        "Best C": grid.best_params_["logistic__C"],
        "CV Accuracy": grid.best_score_ })
l1_cv_results = pd.DataFrame(l1_results)
display(l1_cv_results.sort_values("CV Accuracy", ascending=False).head(10))


# testing 
X_train_l1, y_train_l1 = extract_segment_features(train_files, 19)
X_test_l1, y_test_l1 = extract_segment_features(test_files, 19)

best_l1_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1,
        max_iter=5000 ))  ])

best_l1_model.fit(X_train_l1, y_train_l1)
y_pred_l1 = best_l1_model.predict(X_test_l1)
y_prob_l1 = best_l1_model.predict_proba(X_test_l1)[:,1]

print("Test Accuracy:", round(accuracy_score(y_test_l1, y_pred_l1), 4))
print("Test Error:", round(1 - accuracy_score(y_test_l1, y_pred_l1), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test_l1, y_pred_l1))
print("Test AUC:", round(roc_auc_score(y_test_l1, y_prob_l1), 4))

# ===== Notebook cell 19 =====
#4a vii 
from sklearn.utils import resample

X_train, y_train = extract_segment_features(train_files, best_l)
train_data = X_train.copy()
train_data["label"] = y_train.values

bending = train_data[train_data["label"] == 1]
non_bending = train_data[train_data["label"] == 0]
non_bending_sample = resample(
    non_bending,
    replace=False,
    n_samples=len(bending),
    random_state=42 )

#balanced log reg
balanced_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, C=1e6),
        n_features_to_select=best_p,
        step=0.2 )),
    ("logistic", LogisticRegression(
        max_iter=1000,
        C=1e6 ))  ])

balanced_data = pd.concat([bending, non_bending_sample])
X_balanced = balanced_data.drop(columns=["label"])
y_balanced = balanced_data["label"]

balanced_model.fit(X_balanced, y_balanced)
balanced_pred = balanced_model.predict(X_train)
balanced_prob = balanced_model.predict_proba(X_train)[:,1]

print("Confusion Matrix:")
print(confusion_matrix(y_train, balanced_pred))
print("AUC:", round(roc_auc_score(y_train, balanced_prob), 4))


# adjustments 
from scipy.special import expit, logit

original_p = y_train.mean()
sample_p = y_balanced.mean()
intercept_adjustment = logit(original_p) - logit(sample_p)
adjusted_prob = expit(logit(np.clip(balanced_prob, 1e-10, 1-1e-10)) + intercept_adjustment)

adjusted_pred = (adjusted_prob >= 0.5).astype(int)
print("Adjusted Confusion Matrix:\n", confusion_matrix(y_train, adjusted_pred))
print("intercept adjustment:", round(intercept_adjustment, 4))
print("adjusted AUC", round(roc_auc_score(y_train, adjusted_prob), 4))

#4a vii ROC
fpr, tpr, thresholds = roc_curve(y_train, adjusted_prob)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_train, adjusted_prob):.4f}")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Case-Control Logistic Regression ROC Curve")
plt.legend()
plt.show()

# ===== Notebook cell 21 =====
#4c i 
def extract_multiclass_features(files, l):
    rows = []
    labels = []
    for file in files:
        df = load_arem_file(file)
        sensor_data = df.iloc[:, 1:]
        row = {}
        for i, col in enumerate(sensor_data.columns, start=1):
            values = sensor_data[col]
            segments = np.array_split(values.to_numpy(), l)
            for j, segment in enumerate(segments, start=1):
                row[f"sensor_{i}_part{j}_min"] = np.min(segment)
                row[f"sensor_{i}_part{j}_max"] = np.max(segment)
                row[f"sensor_{i}_part{j}_mean"] = np.mean(segment)
                row[f"sensor_{i}_part{j}_median"] = np.median(segment)
                row[f"sensor_{i}_part{j}_std"] = np.std(segment, ddof=1)
                row[f"sensor_{i}_part{j}_q1"] = np.percentile(segment, 25)
                row[f"sensor_{i}_part{j}_q3"] = np.percentile(segment, 75)
        rows.append(row)
        labels.append(file.parent.name)
    X = pd.DataFrame(rows)
    y = pd.Series(labels)
    return X, y


# multiclass labels verification
X_train_multi, y_train_multi = extract_multiclass_features(train_files, 1)
print("Training shape:", X_train_multi.shape)
print("Class distribution:")
print(y_train_multi.value_counts())

# more CV
#4c i cross validation
from sklearn.model_selection import GridSearchCV

cv_multi = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42)

multi_results = []
C_values = [0.01,0.1,1,10]

for l in range(1,21):
    X,y = extract_multiclass_features(train_files, l)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            penalty="l1",
            solver="saga",
            max_iter=1000,
            tol=0.01,
            random_state=42 ))  ])

    grid = GridSearchCV(
        model,
        {"logistic__C": C_values},
        cv=cv_multi,
        scoring="accuracy",
        n_jobs=-1 )

    grid.fit(X,y)

    multi_results.append({
        "l": l,
        "Best C": grid.best_params_["logistic__C"],
        "CV Accuracy": grid.best_score_ })

    print("Completed l =", l)

multi_cv_results = pd.DataFrame(multi_results)
display(
    multi_cv_results.sort_values(
        "CV Accuracy",
        ascending=False ).head(10))

# ===== Notebook cell 22 =====
#4c i testing portion 
best_l_multi = 1
best_C_multi = 1

X_train_multi, y_train_multi = extract_multiclass_features(train_files, best_l_multi)
X_test_multi, y_test_multi = extract_multiclass_features(test_files, best_l_multi)

multi_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=best_C_multi,
        max_iter=1000,
        tol=0.01,
        random_state=42 ))  ])

multi_model.fit(X_train_multi, y_train_multi)
y_pred_multi = multi_model.predict(X_test_multi)
y_prob_multi = multi_model.predict_proba(X_test_multi)

print("Test Accuracy:", round(accuracy_score(y_test_multi, y_pred_multi), 4))
print("Test Error:", round(1 - accuracy_score(y_test_multi, y_pred_multi), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test_multi, y_pred_multi,
    labels=multi_model.named_steps["logistic"].classes_))

# ===== Notebook cell 24 =====
#4c i multiclass ROC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

classes = multi_model.named_steps["logistic"].classes_
y_test_bin = label_binarize(y_test_multi, classes=classes)
plt.figure(figsize=(8,6))

for i, activity in enumerate(classes):
    fpr, tpr, thresholds = roc_curve(
        y_test_bin[:,i],
        y_prob_multi[:,i] )
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{activity} (AUC = {roc_auc:.4f})")

plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass Logistic Regression ROC Curves")
plt.legend()
plt.show()

# ===== Notebook cell 26 =====
#4c ii Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

gaussian_model = GaussianNB()
gaussian_model.fit(X_train_multi, y_train_multi)
gaussian_pred = gaussian_model.predict(X_test_multi)

print("Gaussian NB Test Accuracy:", round(accuracy_score(y_test_multi, gaussian_pred), 4))
print("Gaussian NB Test Error:", round(1 - accuracy_score(y_test_multi, gaussian_pred), 4))
print("Gaussian NB Confusion Matrix:")
print(confusion_matrix(y_test_multi, gaussian_pred, labels=classes))


# 4c ii continuied 
#4c ii Multinomial Naive Bayes
from sklearn.preprocessing import MinMaxScaler
multinomial_model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("naive_bayes", MultinomialNB())  ])
multinomial_model.fit(X_train_multi, y_train_multi)
multinomial_pred = multinomial_model.predict(X_test_multi)

print("Multinomial NB Test Accuracy:", round(accuracy_score(y_test_multi, multinomial_pred), 4))
print("Multinomial NB Test Error:", round(1 - accuracy_score(y_test_multi, multinomial_pred), 4))
print("Multinomial NB Confusion Matrix:")
print(confusion_matrix(y_test_multi, multinomial_pred, labels=classes))


# l=1 CV 
from sklearn.model_selection import cross_val_score
nb_results = []

for l in range(1,21):
    X,y = extract_multiclass_features(train_files, l)

    gaussian_model = GaussianNB()
    gaussian_scores = cross_val_score(
        gaussian_model,
        X,
        y,
        cv=cv_multi,
        scoring="accuracy" )

    multinomial_model = Pipeline([
        ("scaler", MinMaxScaler()),
        ("naive_bayes", MultinomialNB()) ])

    multinomial_scores = cross_val_score(
        multinomial_model,
        X,
        y,
        cv=cv_multi,
        scoring="accuracy" )

    nb_results.append({
        "l": l,
        "Gaussian CV Accuracy": gaussian_scores.mean(),
        "Multinomial CV Accuracy": multinomial_scores.mean() })

nb_cv_results = pd.DataFrame(nb_results)
display(nb_cv_results.sort_values(
    "Gaussian CV Accuracy",
    ascending=False ).head(10))

#best gaussian
best_gaussian = nb_cv_results.sort_values(
    "Gaussian CV Accuracy",
    ascending=False ).iloc[0]

best_multinomial = nb_cv_results.sort_values(
    "Multinomial CV Accuracy",
    ascending=False ).iloc[0]

print("Best Gaussian l:", int(best_gaussian["l"]))
print("Gaussian CV Accuracy:", round(best_gaussian["Gaussian CV Accuracy"], 4))
print("Best Multinomial l:", int(best_multinomial["l"]))
print("Multinomial CV Accuracy:", round(best_multinomial["Multinomial CV Accuracy"], 4))

# ===== Notebook cell 27 =====
#4c ii final steps 

# Gaussian NB
X_train_g, y_train_g = extract_multiclass_features(train_files, 1)
X_test_g, y_test_g = extract_multiclass_features(test_files, 1)

gaussian_model = GaussianNB()
gaussian_model.fit(X_train_g, y_train_g)
gaussian_pred = gaussian_model.predict(X_test_g)

print("Gaussian NB Test Accuracy:", round(accuracy_score(y_test_g, gaussian_pred), 4))
print("Gaussian NB Test Error:", round(1 - accuracy_score(y_test_g, gaussian_pred), 4))
print("Gaussian NB Confusion Matrix:")
print(confusion_matrix(y_test_g, gaussian_pred, labels=classes))


# Multinomial NB
X_train_m, y_train_m = extract_multiclass_features(train_files, 5)
X_test_m, y_test_m = extract_multiclass_features(test_files, 5)

multinomial_model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("naive_bayes", MultinomialNB())  ])

multinomial_model.fit(X_train_m, y_train_m)
multinomial_pred = multinomial_model.predict(X_test_m)

print("Multinomial NB Test Accuracy:", round(accuracy_score(y_test_m, multinomial_pred), 4))
print("Multinomial NB Test Error:", round(1 - accuracy_score(y_test_m, multinomial_pred), 4))
print("Multinomial NB Confusion Matrix:")
print(confusion_matrix(y_test_m, multinomial_pred, labels=classes))